# Propensity Analysis – Feature Accretion
This notebook visualizes novelty, repetition, truncation, and format adherence to identify the usefulness→repetition cliff.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

DATA_PATH = Path('experiments/outputs/mistral_propensity_features.jsonl')
records = [json.loads(line) for line in DATA_PATH.read_text().splitlines() if line.strip()]
df = pd.DataFrame([r for r in records if r.get('record_type') == 'prompt_response'])
summaries = pd.DataFrame([r for r in records if r.get('record_type') == 'run_summary'])
df.head()

In [ ]:
summaries

## Key signals at a glance

In [ ]:
metrics = {
    'steps': len(df),
    'runs': df['run_id'].nunique(),
    'avg_novelty': df['novelty_score'].mean(),
    'avg_similarity': df['similarity_to_prev'].mean(),
    'avg_repetition_rate': df['repetition_rate'].mean(),
    'hit_max_rate': df['hit_max_gen_tokens'].mean(),
    'format_ok_rate': df['format_ok'].mean() if 'format_ok' in df else None,
    'missing_feature_rate': df['feature_name'].isna().mean() if 'feature_name' in df else None,
}
metrics

## Novelty over time (per run)

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=df, x='step', y='novelty_score', hue='run_id', marker='o')
plt.axhline(0.25, color='red', linestyle='--', label='novelty threshold')
plt.title('Novelty score by step')
plt.legend()
plt.show()

## Repetition vs Similarity

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.lineplot(data=df, x='step', y='repetition_rate', hue='run_id', ax=ax[0])
ax[0].set_title('Repetition rate')
sns.lineplot(data=df, x='step', y='similarity_to_prev', hue='run_id', ax=ax[1])
ax[1].set_title('Similarity to previous')
plt.tight_layout()
plt.show()

## Breakdown score and truncation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.lineplot(data=df, x='step', y='breakdown_score', hue='run_id', ax=ax[0])
ax[0].set_title('Breakdown score')
sns.lineplot(data=df, x='step', y='hit_max_gen_tokens', hue='run_id', ax=ax[1])
ax[1].set_title('Hit max gen tokens (rate)')
plt.tight_layout()
plt.show()

## Feature ledger growth

In [ ]:
if 'feature_ledger_count' in df:
    plt.figure(figsize=(10,4))
    sns.lineplot(data=df, x='step', y='feature_ledger_count', hue='run_id', marker='o')
    plt.title('Feature ledger count')
    plt.show()

## Format adherence (if enabled)

In [ ]:
if 'format_ok' in df:
    plt.figure(figsize=(6,4))
    sns.barplot(x=['format_ok'], y=[df['format_ok'].mean()])
    plt.ylim(0,1)
    plt.title('Format adherence rate')
    plt.show()

## Top missing format fields

In [ ]:
if 'format_missing' in df:
    missing = df['format_missing'].dropna().explode()
    if not missing.empty:
        missing.value_counts().plot(kind='bar', figsize=(6,4), title='Missing format fields')
        plt.show()